# Conversational RAG with Contextual Memory (Lecture 30)

Traditional RAG treats each query in isolation. In real user interactions, questions contain ambiguous references and pronouns (e.g., 'What are its types?', 'How does it work?').
This notebook implements:
1. **Contextualizing Questions**: Rephrasing follow-up queries using conversation history (`create_history_aware_retriever`).
2. **Grounded QA**: Answering questions with retrieved documents (`create_stuff_documents_chain`).
3. **Full Multi-Turn Dialogue**: Maintaining chat history across conversational turns.

## 1. Setup & Corpus Preparation
Initialize ChromaDB with multi-domain technical knowledge.

In [1]:
import os
import shutil
import tempfile
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

try:
    from langchain.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()
temp_dir = tempfile.mkdtemp(prefix="chroma_memory_")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

corpus = [
    "Deep learning is a subfield of artificial intelligence based on representation learning with artificial neural networks.",
    "The primary architectures in deep learning include Convolutional Neural Networks (CNNs) for vision, Recurrent Neural Networks (RNNs) for sequential data, and Transformers for attention-driven tasks.",
    "Transformers use multi-head self-attention mechanisms to process tokens simultaneously rather than recurrence.",
]

from langchain_core.documents import Document
docs = [Document(page_content=text, metadata={"id": i}) for i, text in enumerate(corpus)]
vector_store = Chroma.from_documents(docs, embeddings, persist_directory=temp_dir)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("ChromaDB and Groq LLM initialized.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB and Groq LLM initialized.


## 2. History-Aware Retriever Construction
Build a contextualizing retriever that resolves pronouns using chat history.

In [2]:
contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
print("History-aware retriever constructed.")

History-aware retriever constructed.


## 3. Assembling the Conversational QA Chain
Combine the contextual retriever with the document answering chain.

In [3]:
qa_system_prompt = """You are an assistant for question-answering tasks. \
Use the following pieces of retrieved context to answer the question. \
If you do not know the answer, say that you do not know. Keep answers concise.\n\n{context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
print("Conversational RAG chain assembled.")

Conversational RAG chain assembled.


## 4. Multi-Turn Conversational Testing
Execute Turn 1 (initial concept) followed by Turn 2 (pronoun reference).

In [4]:
chat_history = []

# Turn 1
q1 = "What is deep learning?"
res1 = rag_chain.invoke({"input": q1, "chat_history": chat_history})
print(f"User: {q1}")
print(f"AI: {res1['answer'].strip()}\n")

chat_history.append(HumanMessage(content=q1))
chat_history.append(AIMessage(content=res1["answer"]))

# Turn 2: uses pronoun 'its'
q2 = "What are its primary architectures?"
res2 = rag_chain.invoke({"input": q2, "chat_history": chat_history})
print(f"User: {q2}")
print(f"AI: {res2['answer'].strip()}\n")

assert any("neural" in d.page_content.lower() for d in res2["context"]), "Context must include deep learning architectures"
assert "cnn" in res2["answer"].lower() or "transformer" in res2["answer"].lower(), "Answer must identify architectures"

# Cleanup
shutil.rmtree(temp_dir, ignore_errors=True)
print("Conversational memory verification passed.")

User: What is deep learning?
AI: Deep learning is a subfield of artificial intelligence that uses artificial neural networks to learn representations of data. It focuses on training multi‑layer (deep) models that automatically discover features from raw inputs. The main architectures are:

- **Convolutional Neural Networks (CNNs)** for visual data  
- **Recurrent Neural Networks (RNNs)** for sequential data  
- **Transformers** for attention‑driven tasks such as language and vision.



User: What are its primary architectures?
AI: The main deep‑learning architectures are:

1. **Convolutional Neural Networks (CNNs)** – for image and vision tasks.  
2. **Recurrent Neural Networks (RNNs)** – for sequential or time‑series data.  
3. **Transformers** – for attention‑based tasks such as natural language processing and many vision applications.

Conversational memory verification passed.
